# Adding new functionality
or more verbosely:
# Scenario 2: Adding new dimensions from alternate data streams and providing entirely new functionality

If you want to add new dimensions to an `xsnowDataset` or more generally extend the xsnow functionality beyond a single method, write your own extension class. `xsnow` has everything prepared to make this very straightforward for you. As for a *scenario-1-extension*, write your class in a python module and decide whether you want to keep this module private to yourself or host it in a public repository. 

Here is a cheat sheet for the steps you have to take. You will find more detailed explanations and a demonstration further below:

```{warning}

Coming soon. In the meantime, look at the template [Hazard chart extension](../../api/_generated/xsnow.extensions.hazard_chart) and the `DatasetDecorator` class in [the core module](../../api/_generated/xsnow.core). 
```


```{admonition} Recipe
:class: note

 1. Import: `from xsnow import DatasetDecorator`
 2. Define your extension class: e.g., `class EnsembleFX(DatasetDecorator):`
 3. Define your class methods (and possibly generic functions)
 4. Whenever a function returns an object of your new class, `_rewrap()` the newly generated dataset

```

Regarding 1. and 2.)

 * It is important that you define your class as a *subclass* of the [`DatasetDecorator`](../../api/_generated/xsnow.core). This allows `xsnow` to *configure* your class to feel and behave like an `xsnowDataset`, while allowing multiple extensions to be enchained in custom order.

Regarding 3.)

 * Code all functionality you need and want. Prepend private methods or helper functions with an underscore (e.g., `_my_private_helper`).

 Regarding 4.)

 * Rewrapping is important to ensure different extensions can be enchained. Use the pattern `xs_out = self._rewrap(xs_modified)`.

 *Scenario-2-extensions* can look quite different. Therefore, the next two sections demonstrate two extensions that extend the `xsnow` functionality in their own ways.

## Example: Ensemble-forecast research extension---new data streams and dimensions

The [ensemble forecasts extension](../../api/_generated/xsnow.extensions.ensemble_forecasts) aims to facilitate research on the performance of forecasts with different lead times and from different model realizations such as deterministic or ensemble members. As such, it provides a special read routine that parses a defined directory structure into the dimensions `realization` and model `run`. This read routine is actually the heart of the extension, while the `EnsembleFx` class does not do anything except put its label onto the resulting dataset for consistent naming.

### Excerpt from implementation

In [ ]:
from xsnow import DatasetDecorator

class EnsembleFX(DatasetDecorator):
    """
    Decorator that enriches an xsnowDataset with run context and leadtimes.

    Dimensions/coordinates guaranteed after ``read_ensemble_fx``:
    - ``run`` (string) with attrs including optional ``timezone``.
    - ``realization`` (string) describing the ensemble member label.
    - ``run_start`` coordinate on ``run`` (datetime64[ns], tz in attrs; NaT if unknown).
    - ``leadtime`` coordinate on ``time`` (float hours from run_start to valid time).

    All existing xsnowDataset API remains available via inheritance.
    """
    # note that the class is basically empty (no methods, etc)



# note that this is not a class method, but a module-level function
def read_ensemble_fx(
    source: Union[str, Path],
    # < more parameters >
) -> Optional[EnsembleFX]:
    """
    Read a forecast collection into an ``EnsembleFX`` dataset.

    Layout: ``source/{run}/{member}/{station}.{smet|pro|nc}``. Runs and members are
    derived from folder names; station IDs from filenames. Only requested runs,
    members, and filename bases are read to keep I/O minimal.

    < ... >

    Parameters
    ----------
     < ... >

    Returns
    -------
    EnsembleFX or None
        Decorated dataset when data were found; otherwise ``None``.
    """
    
    # < iterate through directory tree and read >

    # < concatenate individual datasets >

    xs_out = EnsembleFX(xs_combined)

    return xs_out


### Demo application

In [ ]:
import xsnow
from xsnow.extensions.ensemble_forecasts import read_ensemble_fx



INFO:xsnow.xsnow_io:Loading 2 datasets eagerly with 13 workers...


In [3]:
# Apply the extension
sf = SnowFlavor(xs).predict_flavor(model="progressive")
print(sf)

<xsnowDataset>
  Locations:  1
  Timestamps: 381 (2024-01-17--2024-02-02)
  Profiles:   NotImplementedYet

    employing the <xarray.Dataset> Size: 3MB
    Dimensions:                   (location: 1, time: 381, slope: 1,
                                   realization: 1, layer: 59, flavor_model: 1,
                                   flavor_component: 2)
    Coordinates: (12/13)
      * time                      (time) datetime64[ns] 3kB 2024-01-17T16:00:00 ....
      * slope                     (slope) int64 8B 0
      * realization               (realization) int64 8B 0
      * layer                     (layer) int64 472B 0 1 2 3 4 5 ... 54 55 56 57 58
      * location                  (location) <U29 116B 'Kasererwinkl_C6'
        latitude                  (location) float64 8B 47.1
        ...                        ...
        altitude                  (location) float64 8B 1.681e+03
        inclination               (location, slope) int64 8B 16
        azimuth                   (

In [5]:
assert sf.data is not xs.data                      # non-destructive: new dataset
assert sf.base is not xs                           # new base wrapper rebuilt
assert sf.base.data is sf.data                     # synchronized
# _ = sf.sizes; _ = sf.dims; _ = sf.coords           # all show flavor dims
# print(sf)                                          # also shows flavor dims


In [14]:
sf.sel(flavor_component='taste').isel(time=-1, slope=0, realization=0)['flavor']

<xarray.DataArray 'flavor' (flavor_model: 1, location: 1, layer: 59)> Size: 236B
array([[[7.7804604, 7.7788806, 7.774929 , 7.7701783, 6.0143986,
         6.006734 , 6.000338 , 5.992653 , 5.984957 , 5.9785347,
         5.972105 , 5.9656677, 5.9605126, 5.955353 , 5.951479 ,
         5.947603 , 5.5942783, 5.5956526, 7.7804604, 6.005455 ,
         6.0118446, 6.0143986, 6.016951 , 6.027149 , 7.785979 ,
         7.771763 , 4.9676347, 5.053015 , 5.0738363, 5.0708656,
         5.064921 , 3.9709892, 3.9558167, 3.938933 , 3.9203305,
         3.9033911, 3.8830285, 3.8507106, 3.8473034, 1.213983 ,
         3.840485 , 4.423912 , 3.4186254, 4.651004 , 5.11973  ,
         4.899542 , 4.789339 , 4.4029765, 4.407812 , 4.2419786,
         4.069914 , 4.11487  , 4.146386 , 3.409791 , 2.5579147,
         2.9021406, 2.6808558, 2.4998798, 1.518172 ]]], dtype=float32)
Coordinates: (12/13)
    time              datetime64[ns] 8B 2024-02-02T12:00:00
    slope             int64 8B 0
    realization       int64 8B 0
  * layer             (layer) int64 472B 0 1 2 3 4 5 6 ... 52 53 54 55 56 57 58
  * location          (location) <U29 116B 'Kasererwinkl_C6'
    latitude          (location) float64 8B 47.1
    ...                ...
    altitude          (location) float64 8B 1.681e+03
    inclination       (location) int64 8B 16
    azimuth           (location) int64 8B 270
    z                 (location, layer) float32 236B -101.5 -99.71 ... -2.16 0.0
  * flavor_model      (flavor_model) object 8B 'progressive'
    flavor_component  <U11 44B 'taste'
Attributes:
    long_name:    snow flavor components
    description:  Predicted flavor components derived from density and grain_...
    components:   taste, consistency
    model:        progressive
    units:        arb